# Manipulasi Data Gym Members Exercise Tracking

Notebook ini menambahkan dua kolom turunan yang dibutuhkan aplikasi NutriFit:

| Kolom | Diturunkan dari |
| --- | --- |
| `Activity_Level` | frekuensi latihan, durasi sesi, kalori terbakar |
| `Fitness_Goal` | BMI dan persentase lemak tubuh |

**Alur berkas — keduanya wajib ada, bukan berkas kembar:**

```
data/gym_members_exercise_tracking.csv   (masukan, 15 kolom, mentah)
                 |
                 v   notebook ini
data/gym_members.csv                     (keluaran, 17 kolom, dipakai aplikasi)
```

`gym_members.csv` adalah berkas yang dibaca `schema_data/import_csv_to_db.py`
untuk mengisi tabel `gym_members`, lalu dipakai K-Prototypes untuk membentuk
klaster profil anggota. Berkas masukannya jangan dihapus: tanpa itu notebook ini
tidak bisa dijalankan ulang dan asal-usul kedua kolom turunan tidak bisa
dibuktikan saat pengujian.

Jalankan cell dari atas ke bawah saat dataset ingin diproses.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
DATA_PATH = Path("data/gym_members_exercise_tracking.csv")
OUTPUT_PATH = Path("data/gym_members.csv")

# Berkas masukan pernah hilang dari proyek dan membuat notebook ini mati di cell
# pertama. Pesan di bawah menerangkan cara memulihkannya, karena kolom turunan
# hanya DITAMBAHKAN -- keluarannya masih memuat seluruh kolom mentah.
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Berkas masukan tidak ditemukan: {DATA_PATH}\n"
        f"Pulihkan dari keluarannya (kolom mentah masih utuh di sana):\n"
        f'    pd.read_csv("{OUTPUT_PATH}").drop(columns=["Activity_Level", "Fitness_Goal"])'
        f'.to_csv("{DATA_PATH}", index=False)'
    )

df = pd.read_csv(DATA_PATH)
print(f"{len(df)} baris x {len(df.columns)} kolom")
df.head()

973 baris x 15 kolom


,Age,Gender,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Workout_Type,Fat_Percentage,Water_Intake (liters),Workout_Frequency (days/week),Experience_Level,BMI
0,56,Male,88.3,1.71,180,157,60,1.69,1313.0,Yoga,12.6,3.5,4,3,30.20
1,46,Female,74.9,1.53,179,151,66,1.30,883.0,HIIT,33.9,2.1,4,2,32.00
2,32,Female,68.1,1.66,167,122,54,1.11,677.0,Cardio,33.4,2.3,4,2,24.71
3,25,Male,53.2,1.70,190,164,56,0.59,532.0,Strength,28.8,2.1,3,1,18.41
4,38,Male,46.1,1.79,188,158,68,0.64,556.0,Strength,29.2,2.8,3,1,14.39


## 1. Validasi Kolom Wajib

In [3]:
required_columns = [
    "Workout_Frequency (days/week)",
    "Session_Duration (hours)",
    "Calories_Burned",
    "BMI",
    "Fat_Percentage",
]

missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_columns}")

## 2. Tambah Kolom Tingkat Aktivitas

`Activity_Level` dibuat dari kombinasi frekuensi latihan per minggu, durasi sesi, dan kalori terbakar.

In [4]:
def assign_tingkat_aktivitas(row):
    workout_frequency = row["Workout_Frequency (days/week)"]
    session_duration = row["Session_Duration (hours)"]
    calories_burned = row["Calories_Burned"]

    score = 0

    if workout_frequency <= 1:
        score += 1
    elif workout_frequency <= 3:
        score += 2
    elif workout_frequency <= 5:
        score += 3
    else:
        score += 4

    if session_duration >= 1.5:
        score += 1
    elif session_duration >= 1.0:
        score += 0.5

    if calories_burned >= 900:
        score += 1
    elif calories_burned >= 500:
        score += 0.5

    if score <= 1.5:
        return "Low"
    if score <= 3.0:
        return "Medium"
    if score <= 4.5:
        return "High"
    return "Very High"


df["Activity_Level"] = df.apply(assign_tingkat_aktivitas, axis=1)
df["Activity_Level"].value_counts()

Activity_Level
High         403
Medium       377
Very High    193
Name: count, dtype: int64

> **Kategori `Low` tidak muncul, dan itu bukan bug.** Skor terendah yang bisa
> dicapai baris mana pun di dataset ini adalah 2,0 (frekuensi latihan terendah
> 2 hari/minggu, tanpa bonus durasi maupun kalori), sedangkan ambang `Low`
> adalah skor ≤ 1,5 yang hanya tercapai bila frekuensi ≤ 1 hari/minggu.
> Dataset ini memang tidak memuat anggota yang berlatih 0–1 hari/minggu, jadi
> pemetaannya menghasilkan tiga kategori: `Medium`, `High`, `Very High`.

## 3. Tambah Kolom Tujuan Kebugaran

`Fitness_Goal` dibuat dari BMI dan persentase lemak tubuh.

In [5]:
def assign_tujuan_kebugaran(row):
    bmi = row["BMI"]
    fat_percentage = row["Fat_Percentage"]

    if bmi >= 25 or fat_percentage >= 28:
        return "Lose Weight"
    if bmi < 18.5:
        return "Gain Weight"
    return "Maintain Weight"


df["Fitness_Goal"] = df.apply(assign_tujuan_kebugaran, axis=1)
df["Fitness_Goal"].value_counts()

Fitness_Goal
Lose Weight        641
Maintain Weight    235
Gain Weight         97
Name: count, dtype: int64

## 4. Cek Hasil Manipulasi

In [6]:
selected_columns = [
    "Age",
    "Gender",
    "Weight (kg)",
    "Height (m)",
    "Workout_Frequency (days/week)",
    "Session_Duration (hours)",
    "Calories_Burned",
    "Fat_Percentage",
    "BMI",
    "Activity_Level",
    "Fitness_Goal",
]

df[selected_columns].head(10)

,Age,Gender,Weight (kg),Height (m),Workout_Frequency (days/week),Session_Duration (hours),Calories_Burned,Fat_Percentage,BMI,Activity_Level,Fitness_Goal
0,56,Male,88.3,1.71,4,1.69,1313.0,12.6,30.20,Very High,Lose Weight
1,46,Female,74.9,1.53,4,1.30,883.0,33.9,32.00,High,Lose Weight
2,32,Female,68.1,1.66,4,1.11,677.0,33.4,24.71,High,Lose Weight
3,25,Male,53.2,1.70,3,0.59,532.0,28.8,18.41,Medium,Lose Weight
4,38,Male,46.1,1.79,3,0.64,556.0,29.2,14.39,Medium,Lose Weight
5,56,Female,58.0,1.68,5,1.59,1116.0,15.5,20.55,Very High,Maintain Weight
6,36,Male,70.3,1.72,3,1.49,1385.0,21.3,23.76,High,Maintain Weight
7,40,Female,69.7,1.51,3,1.27,895.0,30.6,30.57,Medium,Lose Weight
8,28,Male,121.7,1.94,4,1.03,719.0,28.9,32.34,High,Lose Weight
9,28,Male,101.8,1.84,3,1.08,808.0,29.7,30.07,Medium,Lose Weight


In [7]:
summary = pd.crosstab(df["Activity_Level"], df["Fitness_Goal"])
summary

Fitness_Goal,Gain Weight,Lose Weight,Maintain Weight
Activity_Level,,,
High,41,293,69
Medium,46,267,64
Very High,10,81,102


## 5. Simpan Dataset Hasil Manipulasi

Hasil disimpan ke `data/gym_members.csv`, berkas yang dibaca aplikasi. Dataset
mentahnya (`gym_members_exercise_tracking.csv`) tidak tertimpa.

In [8]:
# lineterminator="\n" dipasang eksplisit supaya notebook ini idempoten: tanpa itu
# pandas memakai akhir baris bawaan sistem (CRLF di Windows, LF di macOS/Linux),
# sehingga menjalankan ulang notebook di komputer berbeda menandai SELURUH 974
# baris sebagai berubah walaupun tidak ada satu pun nilai yang berbeda.
df.to_csv(OUTPUT_PATH, index=False, lineterminator="\n")
print(f"Dataset hasil manipulasi disimpan ke: {OUTPUT_PATH}")
print(f"{len(df)} baris x {len(df.columns)} kolom")

Dataset hasil manipulasi disimpan ke: data\gym_members.csv
973 baris x 17 kolom
